# Git 分支与协作

## 分支与协作是什么？

Git 的分支是一个指向某次 commit 的哈希值的轻量指针。

### main

Git 仓库初始化时会自动创建一个默认分支，通常叫 `main`（旧版本叫 `master`）。`main` 指向当前开发线上的最新提交。

### HEAD：

HEAD 是指向**你当前所在的位置**的特殊指针。

## 怎么用？

### git branch：查看与创建分支

In [ ]:
git branch                 # 列出所有本地分支，当前分支前有 * 号
git branch feature-login   # 创建一个名为 feature-login 的新分支
git branch -d feature-login # 删除分支（-d = delete，安全删除：如果分支未合并会提示）
git branch -D feature-login # 强制删除（-D = 不管合没合并，直接删）

> 分支命名习惯：功能分支通常用 `feature/xxx`（如 `feature/login-page`），bug 修复用 `fix/xxx`（如 `fix/typo`），紧急补丁用 `hotfix/xxx`。用斜杠可以分组，视觉上更清晰。

### git switch / git checkout：切换分支

In [ ]:
git switch feature-login      # 切换到 feature-login 分支（Git 2.23+，推荐）
git switch -c feature-signup  # 创建并切换到新分支（-c = create）
git switch -                  # 切回上一个分支（像电视遥控器的「返回」键）

# 旧写法（老教程常见，功能相同）
git checkout feature-login
git checkout -b feature-signup  # 创建并切换到新分支

### git merge：合并分支

假设你在 `feature-login` 上开发完了，想把代码合并回 `main`：

In [ ]:
git switch main             # 1. 先回到 main
git merge feature-login     # 2. 把 feature-login 合并到 main

merge 有两种情况：

**情况一：快进合并（Fast-forward）**

如果 `main` 在A处分支后没有新提交，Git 只是把 `main` 指针向前滑到 `feature-login` 的最新提交，和新建分支修改代码一样。

```
合并前：
  A ── B ── C  ← feature    ← feature 往前走了两步
  ↑ 
 main                       ← main 还停在 A，没动过

合并后：
  A ── B ── C  ← main, feature
               （main 指针直接滑到 C，没有产生新提交）
```

**情况二：三方合并（Three-way Merge）**

如果在B处分支后两边都有新提交，Git 需要把分支点（B）、main 的最新（D）、feature 的最新（F）三方汇合，生成一个全新的「合并提交」（merge commit）。

```
合并前：
         C ── D  ← main      ← main 往前走了两步
        /
  A ── B                     ← 分叉点B
        \
         E ── F  ← feature   ← feature 也往前走了两步

合并后：
         C ── D 
        /       \
  A ── B         M  → main    ← M 是新创建的合并提交，同时连着 D 和 F
        \       /
         E ── F  → feature
```

M 是一个特殊的提交：它有两个父提交（D 和 F），把两边的改动汇合到一起。这也是为什么叫「三方合并」——Git 需要对比 B（共同祖先）、D（main 的版本）、F（feature 的版本）三方内容来决定最终结果。

---

### 冲突（Conflict）

如果 `main` 和 `feature` 改了同一个文件的同一行，Git 不知道该用哪个版本——这就是**冲突**（conflict：合并时两个版本对同一位置做了不同修改，Git 无法自动决定用哪个）。

此时 Git 会暂停合并，在冲突文件中标记出两边的版本：

```
<<<<<< HEAD
const port = 3000;（main的修改）
=======
const port = 8080;（分支的修改）
>>>>>> feature-login
```

**手动解决冲突**，然后再提交：

```bash
git add .                  # 把解决后的文件加入暂存区
git commit                 # 完成合并提交（不要加 -m，Git 会自动生成合并信息）
```

如果冲突太严重想放弃本次合并：

```bash
git merge --abort          # 放弃合并，回到合并前的状态
```

### git rebase：变基

> merge 保留真实的历史分叉；rebase 把分叉线「熨平」成一条直线。团队通常约定用一种方式。

In [ ]:
git switch feature-login
git rebase main            # 把 feature-login 的提交「搬到」main 最新提交后面

```
rebase 前：
        C ← D ← main
       /
  A ← B
       \
        E ← F ← feature

rebase 后，feature 上原来的提交会被重新应用到 main 最新提交 D 的后面，生成新的提交 E'、F'（D打上E的补丁成为E'，E'打上F的补丁成为F'）
  A ← B ← C ← D ← E' ← F' ← feature（E' 和 F' 是新提交，哈希值变了）
              ↑
             main
```

**rebase 的黄金规则**：永远不要 rebase 已经 push 到远程的公共分支。因为它改写了提交历史（哈希值变了）。

### git remote：管理远程仓库

把本地仓库和一个远程仓库关联起来：

In [ ]:
git remote add origin https://github.com/用户名/项目名.git    # 关联远程仓库，起个别名叫 origin
git remote -v                                                # 查看所有远程仓库的地址
git remote show origin                                       # 查看某个远程仓库的详细信息

> URL（Uniform Resource Locator，统一资源定位符）是互联网上的「地址」。`https://github.com/用户名/项目名.git` 就是一个 URL.

### git push：推送本地提交到远程

In [ ]:
git push origin main                    # 把本地的 main 分支推送到 origin(远端仓库)的 main 分支
git push -u origin main                 # -u = --set-upstream，推送时绑定 main 分支的上下游
git push                                # 绑定上下游后直接推送当前分支
git push origin feature-login           # 推 feature-login 分支（还未绑定上下游）

> 上游（upstream）是「本地分支对应的远程分支」。比如本地 `main` 的上游是 `origin/main`。绑定之后，`git push` 和 `git pull` 就知道已绑定的当前分支往哪推、从哪拉。

### git fetch 与 git pull：从远程获取更新

当别人推送了新代码，你需要把它拉到本地。Git 提供了两个命令，安全程度不同：

In [ ]:
git fetch origin          # 查看 origin 上别人更新了什么（不自动合并，只下载数据）
git pull origin main      # 拉取 main 的最新代码并自动与当前分支合并（= fetch + merge）
git pull --rebase         # 拉取并 rebase（而不是 merge），保持线性历史

### git stash：暂时储存当前修改

你在 `feature-login` 分支上写了一堆代码，还没到能 commit 的程度。突然线上出了 bug，必须立刻切到 `main` 去修。但这些半成品代码挡着你，不让切——`stash` 就是用来解决这个场景的：**把工作区和暂存区的修改临时存起来，让工作区变干净，需要时再取出来。**

In [ ]:
git stash                  # 暂存当前分支所有修改
git stash save "写了一半的登录功能"  # 暂存时加备注（-m 也行）
git stash list             # 查看全局所有暂存记录（Git 自动标注当时所在分支）
git stash pop              # 恢复最近一次暂存的修改到当前分支，并删除那条暂存记录
git stash apply            # 恢复最近一次暂存的修改到当前分支，但保留记录（不删除）
git stash drop             # 删除最近一条暂存记录
git stash clear            # 清空所有暂存这样？

## 典型协作工作流

两个人合作开发一个功能的最简流程：

```bash
# --- 你 ---
git switch -c feature-login       # 从本地 main 创建并切换功能分支
# 写代码...
git add . && git commit -m "完成登录页面" # 提交修改
git push -u origin feature-login  # 推送到远程的 feature-login 分支

# --- 同事 ---
git fetch origin                  # 下载（查看）远程更新
git switch -c feature-login origin/feature-login  # 创建并切换到该分支（拉下你的分支）
# 写代码...
git add . && git commit -m "添加登录验证逻辑"
git push

# --- 你 ---
git pull                         # 拉下同事的更新，并合并
# 如果有冲突 → 解决冲突 → add → commit
# 确认没问题后
git switch main
git merge feature-login          # 合并到主分支
git push origin main             # 推送主分支
git branch -d feature-login      # 清理本地分支
git push origin --delete feature-login  # 清理远程分支
```

---

## 本篇出现的新名词速查

| 名词 | 一句话解释 |
|------|-----------|
| 分支 (Branch) | 指向某次提交的轻量指针，让多条开发线并行互不干扰 |
| HEAD | 指向你当前所在分支的特殊指针 |
| switch | 切换分支，本质是移动 HEAD 指针到另一个分支 |
| 远程仓库 (Remote) | 托管在服务器上的仓库，如 GitHub/GitLab，约定俗成的交换中心 |
| origin | 远程仓库的默认别名，就是给远程地址起的小名 |
| 快进合并 (Fast-forward) | merge 时如果 main 分支没有新提交，直接滑 main 指针，不产生合并提交 |
| 三方合并 (Three-way Merge) | merge 时两边都有新提交，Git 自动（或手动）生成一个合并提交 |
| 冲突 (Conflict) | 两个分支改了同一行，Git 无法自动决定用哪个，需要人工解决 |
| rebase | 变基——把一条分支的提交「摘下来」接到另一条分支后面，保持线性历史 |
| fetch | 从远程下载更新但不合并，安全查看别人改了什么 |
| pull | fetch + merge，一步拉取并合并 |
| push | 把本地提交推送到远程仓库 |
| upstream | 本地分支对应的远程分支，绑定后用 push/pull 就不需要每次指定目标 |
| stash | 把未提交的修改临时存起来，让工作区变干净，需要时再取 |
| URL | 统一资源定位符，互联网上的地址格式 |